In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from matplotlib.lines import Line2D

sys.path.append("/workspace")

from vertexcbf.config_utils import build_constr_fn, build_dynamics, build_model
from vertexcbf.validation import validate_cbf

%reload_ext autoreload
%autoreload 2

WORKSPACE = "/workspace"
DEVICE = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

# Which paper-method group to visualize.  Sub-folders under checkpoints/ and
# figures/ are organized by this group:
#   NO_DATA  — PDE/HJB residual loss only.
#   FC_DATA  — full-control sampling MPC (mppi / cem / icem / random shooting).
#   VRC_DATA — vertex-restricted control search
#              (beam / B&B / stochastic beam / cem_discrete).
METHOD = "VRC_DATA"

# DPI for the rasterized heatmap layer in saved PDFs. Contours, text, axes
# and colorbar stay vector regardless. 200 is plenty for print; bump if needed.
SAVE_DPI = 200

Path(f"figures/{METHOD}").mkdir(parents=True, exist_ok=True)
print(f"Device: {DEVICE}")
print(f"Method group: {METHOD}")


def _load_system(system):
    """Load config, dynamics, model, ground truth, and training validation summary."""
    cfg_path = f"{WORKSPACE}/configs/{system}.yaml"
    ckpt_path = f"{WORKSPACE}/checkpoints/{METHOD}/{system}/final.pt"
    val_path = f"{WORKSPACE}/checkpoints/{METHOD}/{system}/validation.json"
    truth_dir = f"{WORKSPACE}/data/true_values/{system}"

    with open(cfg_path) as f:
        cfg = yaml.safe_load(f)
    dynamics = build_dynamics(cfg["system"], device=DEVICE)
    constr_fn = build_constr_fn(cfg["constraint"])
    model = build_model(cfg["model"], dynamics, device=DEVICE)

    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    with open(val_path) as f:
        val_summary = json.load(f)

    # Optional precomputed ground-truth values on a regular grid.
    # Ground truth is method-independent, so it lives at data/true_values/<system>/
    # (not under a method-group sub-folder).
    values_true = grid_true = None
    if os.path.isfile(f"{truth_dir}/values.npy"):
        values_true = np.load(f"{truth_dir}/values.npy")
        grid_true = np.load(f"{truth_dir}/grid.npy")

    return cfg, dynamics, constr_fn, model, val_summary, values_true, grid_true


def _grid_axis_coords(grid_true):
    """Extract the 1-D coordinate vector along each axis from an ij-meshgrid."""
    nx = grid_true.shape[-1]
    coords = []
    for i in range(nx):
        sel = [0] * nx
        sel[i] = slice(None)
        coords.append(grid_true[tuple(sel) + (i,)])
    return coords


def _slice_ground_truth(values_true, grid_true, slice_axes, slice_fixed_values):
    """Reduce any-D ground truth to a 2-D (tg0, tg1, values_2d) slice.

    For the "other" axes, snaps to the nearest grid index of the requested
    fixed value. Returns None if a required fixed value is missing.
    """
    nx = grid_true.shape[-1]
    if grid_true.ndim == 3:  # already a 2-D slice (n0, n1, nx)
        return grid_true[..., slice_axes[0]], grid_true[..., slice_axes[1]], values_true

    coords = _grid_axis_coords(grid_true)

    # Slice on the lower-indexed axis first so the resulting array is in
    # natural (low_axis, high_axis) order; transpose at the end if needed.
    lo, hi = sorted(slice_axes)
    indexer = [None] * nx
    indexer[lo] = slice(None)
    indexer[hi] = slice(None)
    for i in range(nx):
        if i in slice_axes:
            continue
        if i not in slice_fixed_values:
            return None
        indexer[i] = int(np.argmin(np.abs(coords[i] - slice_fixed_values[i])))

    values_2d = values_true[tuple(indexer)]
    if slice_axes != (lo, hi):
        values_2d = values_2d.T

    tg0, tg1 = np.meshgrid(coords[slice_axes[0]], coords[slice_axes[1]], indexing="ij")
    return tg0, tg1, values_2d


def _build_slice(dynamics, slice_axes, slice_fixed_values, slice_shape, grid_true):
    """Return (g0, g1, slice_states) over the 2-D slice."""
    ax0, ax1 = slice_axes
    other_axes = [i for i in range(dynamics.nx) if i not in slice_axes]

    # Reuse the ground-truth grid only when it already lives on a 2-D slice
    # (shape (n0, n1, nx)). For higher-dim systems we build the slice ourselves.
    if grid_true is not None and grid_true.ndim == 3:
        g0 = grid_true[..., ax0]
        g1 = grid_true[..., ax1]
        slice_states = grid_true.reshape(-1, dynamics.nx).astype(np.float32)
        return g0, g1, slice_states

    missing = [i for i in other_axes if i not in slice_fixed_values]
    if missing:
        raise ValueError(f"slice_fixed_values must specify axes {missing}")

    x_min = dynamics.x_min.cpu().numpy()
    x_max = dynamics.x_max.cpu().numpy()
    n0, n1 = slice_shape
    g0, g1 = np.meshgrid(
        np.linspace(x_min[ax0], x_max[ax0], n0),
        np.linspace(x_min[ax1], x_max[ax1], n1),
        indexing="ij",
    )

    slice_states = np.zeros((n0 * n1, dynamics.nx), dtype=np.float32)
    slice_states[:, ax0] = g0.ravel()
    slice_states[:, ax1] = g1.ravel()
    for i in other_axes:
        slice_states[:, i] = slice_fixed_values[i]
    return g0, g1, slice_states


def _predict_values(model, constr_fn, states_np):
    """V(x) = c(x) - r_theta(x) for a flat (N, nx) array."""
    with torch.no_grad():
        x = torch.as_tensor(states_np, dtype=torch.float32, device=DEVICE)
        return (constr_fn(x) - model(x)).squeeze(-1).cpu().numpy()


def _false_rates(val_summary):
    """Pull (false_safe, false_unsafe) percentages from either validation.json format."""
    sm = val_summary.get("stratified_metrics")
    if sm is not None:
        return sm["predicted_safe"]["false_safe_rate"], sm["predicted_unsafe"]["false_unsafe_rate"]
    cm = val_summary["confusion_matrix"]
    return cm["false_safe"], cm["false_unsafe"]


def visualize(
    system,
    slice_axes,
    slice_axes_labels,
    slice_fixed_values=None,
    slice_shape=(200, 200),
    save=True,
):
    """Visualize a 2-D slice of V(x) (predicted, true, validated) and print training rates."""
    cfg, dynamics, constr_fn, model, val_summary, values_true, grid_true = _load_system(system)
    print(f"System: {dynamics.name}, nx={dynamics.nx}, nu={dynamics.nu}")

    # Build the slice grid and evaluate predicted V and the constraint on it.
    g0, g1, slice_states = _build_slice(
        dynamics, slice_axes, slice_fixed_values or {}, slice_shape, grid_true,
    )
    n0, n1 = g0.shape
    values_pred = _predict_values(model, constr_fn, slice_states).reshape(n0, n1)
    states_t = torch.as_tensor(slice_states, dtype=torch.float32, device=DEVICE)
    constr = constr_fn(states_t).cpu().numpy().reshape(n0, n1)

    # Closed-loop validation on the slice states (used only for the lime contour).
    val_cfg = cfg.get("validation", {}) or {}
    slice_val = validate_cbf(
        dynamics=dynamics,
        states=states_t,
        constr_fn=constr_fn,
        residual_fn=model,
        T=val_cfg["T"],
        dt=val_cfg["dt"],
        return_values=True,
        batch_size=val_cfg.get("batch_size"),
    )
    values_val = slice_val["values"].cpu().numpy().reshape(n0, n1)

    # Reduce any-D ground truth to the chosen 2-D slice.
    gt_slice = None
    if values_true is not None:
        gt_slice = _slice_ground_truth(
            values_true, grid_true, slice_axes, slice_fixed_values or {},
        )

    ax0, ax1 = slice_axes
    x_min = dynamics.x_min.cpu().numpy()
    x_max = dynamics.x_max.cpu().numpy()

    # Plot: background heatmap (rasterized for compact PDFs), then contours
    # in the requested order. rasterized=True applies only to the pcolormesh
    # layer; contours, axes, ticks, labels, legend and colorbar stay vector.
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.pcolormesh(
        g0, g1, values_pred, cmap="plasma", shading="auto", rasterized=True,
    )
    ax.contour(g0, g1, constr, levels=[0], colors="black", linewidths=2.0)

    handles = [Line2D([0], [0], color="black", lw=2.0, label="Constraint")]

    if gt_slice is not None:
        tg0, tg1, values_true_2d = gt_slice
        ax.contour(tg0, tg1, values_true_2d, levels=[0], colors="yellow", linewidths=1.5)
        handles.append(Line2D([0], [0], color="yellow", lw=1.5, label="Ground truth"))

    ax.contour(g0, g1, values_pred, levels=[0],
               colors="dodgerblue", linewidths=1.5, linestyles="--")
    handles.append(Line2D([0], [0], color="dodgerblue", lw=1.5, ls="--", label="Predicted"))

    ax.contour(g0, g1, values_val, levels=[0],
               colors="lime", linewidths=1.5, linestyles="-.")
    handles.append(Line2D([0], [0], color="lime", lw=1.5, ls="-.", label="Validated"))

    ax.legend(handles=handles, loc="best", fontsize=11)
    ax.set_xlabel(slice_axes_labels[0], fontsize=14)
    ax.set_ylabel(slice_axes_labels[1], fontsize=14)
    ax.set_xlim(x_min[ax0], x_max[ax0])
    ax.set_ylim(x_min[ax1], x_max[ax1])
    cbar = plt.colorbar(im, ax=ax, location="right")
    cbar.ax.set_title(r"$V_{\Theta}(x)$")
    plt.tight_layout()
    plt.show()

    if save:
        fig.savefig(
            f"figures/{METHOD}/2d_slice_{system}.pdf",
            bbox_inches="tight",
            dpi=SAVE_DPI,
        )

    # Sign agreement against the precomputed ground truth (when grids match exactly).
    if gt_slice is not None and gt_slice[2].shape == values_pred.shape:
        mse = float(np.mean((values_pred - gt_slice[2]) ** 2))
        sign_agree = float(np.mean(np.sign(values_pred) == np.sign(gt_slice[2])))
        print(f"MSE vs ground truth:     {mse:.6f}")
        print(f"Safe-set sign agreement: {sign_agree * 100:.2f}%")

    # Training-time validation rates loaded straight from validation.json.
    false_safe, false_unsafe = _false_rates(val_summary)
    print(f"False safe (training):   {false_safe:.4f}%")
    print(f"False unsafe (training): {false_unsafe:.4f}%")


In [ ]:
visualize(
    system="double_integrator_1d",
    slice_axes=(0, 1),
    slice_axes_labels=(r"$p$", r"$v$"),
)

In [ ]:
visualize(
    system="inverted_pendulum",
    slice_axes=(0, 1),
    slice_axes_labels=(r"$\theta$", r"$\omega$"),
)

In [ ]:
visualize(
    system="vertical_drone_2d",
    slice_axes=(0, 1),
    slice_axes_labels=(r"$z$", r"$v_z$"),
)

In [ ]:
visualize(
    system="dubins_car",
    slice_axes=(0, 1),
    slice_axes_labels=(r"$p_x$", r"$p_y$"),
    slice_fixed_values={2: -0.8377},
)

In [ ]:
visualize(
    system="double_integrator_2d",
    slice_axes=(0, 1),
    slice_axes_labels=(r"$p_x$", r"$p_y$"),
    slice_fixed_values={2: 2.0, 3: 1.0},
)

In [ ]:
visualize(
    system="cart_pole",
    slice_axes=(0, 1),
    slice_axes_labels=(r"$p$", r"$\theta$"),
    slice_fixed_values={2: 0.0, 3: 0.0},
)

In [ ]:
visualize(
    system="kinematic_bicycle",
    slice_axes=(0, 1),
    slice_axes_labels=(r"$p_x$", r"$p_y$"),
    slice_fixed_values={2: 0.0, 3: 20.0},
)

In [ ]:
visualize(
    system="dynamic_unicycle",
    slice_axes=(0, 1),
    slice_axes_labels=(r"$p_x$", r"$p_y$"),
    slice_fixed_values={2: 0.0, 3: 2.0, 4: 0.0},
)

In [ ]:
visualize(
    system="quadrotor",
    slice_axes=(0, 1),
    slice_axes_labels=(r"$p_x$", r"$p_y$"),
    slice_fixed_values={
        2: 0.0,
        3: 0.0,
        4: 0.0,
        5: 0.0,
        6: 1.0,
        7: 4.0,
        8: 0.0,
        9: 0.0,
        10: 0.0,
        11: 0.0,
        12: 0.0,
    },
)

In [ ]:
visualize(
    system="landing_rocket",
    slice_axes=(0, 1),
    slice_axes_labels=(r"$p_x$", r"$p_z$"),
    slice_fixed_values={2: 1.0, 3: -6.0, 4: -0.2, 5: 0.0, 6: 0.9},
    slice_shape=(100, 100),
)

In [ ]:
visualize(
    system="double_integrator_3d",
    slice_axes=(0, 1),
    slice_axes_labels=(r"$p_x$", r"$p_y$"),
    slice_fixed_values={2: 0.0, 3: 2.0, 4: 2.0, 5: 0.0},
)

In [ ]:
visualize(
    system="manipulator_3dof",
    slice_axes=(1, 2),
    slice_axes_labels=(r"$q_2$", r"$q_3$"),
    slice_fixed_values={0: 0.25, 3: 0.0, 4: -5.0, 5: -0.0},
)

In [ ]:
visualize(
    system="relative_unicycle",
    slice_axes=(0, 1),
    slice_axes_labels=(r"$p_x$", r"$p_y$"),
    slice_fixed_values={2: 3.14159, 3: 1., 4: 1.},
)

In [ ]:
visualize(
    system="quadruped_trunk",
    slice_axes=(1, 2),
    slice_axes_labels=(r"$\phi$", r"$\theta$"),
    slice_fixed_values={
    0: 0.3,
    # 1: 0.0,
    # 2: 0.0,
    3: 0.0,
    4: 0.0,
    5: .0,
    6: -4.0,
    7: -4.0,
    8: 0.0,
} ,
)

In [ ]:
visualize(
    system="auv_6dof",
    slice_axes=(1, 2),
    slice_axes_labels=(r"$p_y$", r"$p_z$"),
    slice_fixed_values={
        0: 0.0,
        # 1: 0.0,
        # 2: 0.0,
        3: 0.0,
        4: -1.0,
        5: 1.0,
        6: 1.8,
        7: 0.0,
        8: -1.8,
        9: 0.0,
        10: 0.0,
        11: 0.0,
    },
)